## **Overview**

This project revisits a scientific-programming exercise on **three-body bond-angle distributions** in model atomic systems. I use it to explore both structural information contained in bond-angle distributions and the numerical choices involved in neighbour identification. The original coursework implementation and accompanying write-up have been reorganised and annotated here as part of my later revision.

To keep it concise this work mainly contains the code. Discussions are made in the explanary text.



## **Objective**
The model is deliberately simple, providing an intuitive setting in which the relationship between atomic arrangement and local three-body geometry can be examined directly. By changing the degree and type of structural order while keeping the underlying analysis consistent, the same framework can be used to observe how bond-angle distributions respond to increasingly different atomic environments. More realistic physical effects can subsequently be incorporated on top of this basic framework.

Specifically, the project considers four related structural scenarios:

1. *General periodic configuration*:
How can local three-body geometry be defined and computed consistently in a finite simulation cell representing an infinite periodic system?

2. *Ordered atomic structure*:
How is structural order reflected in the bond-angle distribution?

3. *Positional disorder / perturbed structure*:
How does the bond-angle distribution change when atoms are displaced from their ideal lattice positions?

4. *Uncorrelated reference system*:
What does the bond-angle distribution look like in the absence of underlying structural order? We discuss the ideal gas configuration for example.

From a computational perspective, the main focus is the **neighbour-identification step**, which forms the numerical bottleneck of the bond-angle calculation. **Four** approaches are investigated: ***brute-force search, a vectorised implementation, KD-tree search, and a cell-linked-list method***. Their results are compared not only for computational efficiency and scalability, but also for numerical consistency, so that improvements in performance do not come at the expense of the accuracy of the calculated bond-angle distributions.




In [4]:
import sys
print(sys.executable)

d:\00_2026_phd_application\Computational_Practice_MSc\.venv\Scripts\python.exe


In [18]:
import numpy as np


def make_cell(a, b, c):
    """
    Construct a 3D periodic simulation cell from three linearly independent cell vectors(lattice vector).
    Lattice vectors defining the size and shape of the simulation cell.
    Cartesian components are given in angstroms (Å).
    Returns cell : ndarray, shape (3, 3)
    Cell vectors stored as rows(row vectors).
    """
    cell = np.array([a, b, c], dtype=float)

    if cell.shape != (3, 3):
        raise ValueError("A 3D cell must contain three 3D vectors.")

    if np.isclose(np.linalg.det(cell), 0.0):
        raise ValueError("Cell vectors must be linearly independent.")

    return cell

### **Periodic boundary conditions**

Periodic boundary conditions are important for the following parts of this project.

At this stage, the simulation cell has already been defined. The next step is to populate the cell with particles. When generating particle positions, we do not want two particles to be placed too close to each other or to overlap. Therefore, when checking the separation between a newly generated particle and the particles already in the cell, periodic boundary conditions need to be considered. This is because simply calculating the Euclidean distance between two Cartesian positions is not always sufficient in a periodic system. Two particles may appear to be far apart inside the primary simulation cell, while their nearest periodic images are actually very close to each other.

The same issue also appears later during neighbour searching. A particle close to one boundary of the simulation cell may be a neighbour of another particle close to the opposite boundary. Therefore, the minimum-image displacement should be used before calculating the interparticle distance.

The *pbc()* function below is therefore used both during particle generation and during neighbour identification.

In [ ]:
def pbc(displacement, cell):
    """
    This function applies periodic boundary conditions to a Cartesian displacement vector
    and finds the shortest equivalent displacement under periodicity, where

    displacement : array_like, shape (3,) Cartesian displacement vector between two atoms.
    cell : ndarray, shape (3, 3) Cell vectors stored as rows.

    Returns
    -------
    ndarray, shape (3,)
        Minimum-image displacement vector in Cartesian coordinates.
    """
    fractional = displacement @ np.linalg.inv(cell)
    fractional -= np.round(fractional)

    return fractional @ cell

Too keep things in general we populate the box with particles at random position(objective 1). To avoid unrealistically close particle pairs, a minimum allowed interparticle separation is introduced during random particle generation.

In [19]:
def generate_random_positions(n_particles, cell, min_distance, seed=None):

    """ The function is used to generate random particles, where
        
        n_particles: number of particles
        cell: simulation cell
        min_distance:the minimum distance between two particles to avoild unphysically overlap or close particle pairs 
    
    """

    rng = np.random.default_rng(seed)
    positions = []

    while len(positions) < n_particles:

        fractional_position = rng.random(3)
        candidate = fractional_position @ cell

        valid = True

        for position in positions:

            displacement = candidate - position
            displacement = pbc(displacement, cell)

            distance = np.linalg.norm(displacement)

            if distance < min_distance:
                valid = False
                break

        if valid:
            positions.append(candidate)

    return np.array(positions)

In [ ]:
def generate_ordered_positions(cell, n_a, n_b, n_c):
    """
    Generate regularly spaced particle positions inside a periodic cell.

    The ordered grid is first constructed in fractional coordinates
    and then transformed into Cartesian coordinates.
    """
    positions = []

    for i in range(n_a):
        for j in range(n_b):
            for k in range(n_c):

                fractional = np.array([
                    i / n_a,
                    j / n_b,
                    k / n_c
                ])

                position = fractional @ cell
                positions.append(position)

    return np.array(positions)

### Generating physical crystal structures with ASE###
The ordered array above is a simple geometric construction. For standard crystal structures, it is more convenient to use established tools such as ASE, which can generate common lattices and supercells directly.